# GPT from scratch — Part 2: Attention

Continues from `gpt.ipynb` (data → tokenizer → batching → trained bigram baseline).
The bigram hit a ~2.5 loss ceiling because it can only condition on the **single current
character**. Attention lets every position gather information from all previous positions —
that's what breaks the ceiling.

**Roadmap this notebook:** weighted-average trick → single-head self-attention (Q/K/V) →
multi-head → transformer block (residual + LayerNorm + feed-forward) → scale up.

The setup below is carried over from Part 1 — run it top-to-bottom, then we build attention.

## Setup (carried over from Part 1)

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', device)

torch 2.12.1 | device: mps


In [3]:
with open('shakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
print('length:', len(text), '| vocab_size:', vocab_size)

length: 1115394 | vocab_size: 65


In [4]:
# Tokenizer (from Part 1)
stoi = {c: n for n, c in enumerate(chars)}
itos = {n: c for n, c in enumerate(chars)}

def encode(s):
    return list(map(lambda c: stoi[c], s))

def decode(l):
    return ''.join(map(lambda i: itos[i], l))

assert decode(encode('hii there')) == 'hii there'

In [5]:
# Corpus -> tensor, 90/10 train/val split (from Part 1)
data = torch.tensor(encode(text), dtype=torch.int64)
n = int(0.9 * data.shape[0])
train_data = data[:n]
val_data = data[n:]
print('train:', len(train_data), '| val:', len(val_data))

train: 1003854 | val: 111540


In [6]:
# Hyperparameters + batching (from Part 1)
block_size = 8   # T: max context length
batch_size = 4   # B: independent sequences per batch

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(0, d.shape[0] - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

xb, yb = get_batch('train')
print('xb', xb.shape, '| yb', yb.shape)

xb torch.Size([4, 8]) | yb torch.Size([4, 8])


## Step 7 — Attention, motivation: "average your predecessors"

Before Q/K/V, the gentlest version of the idea. Right now each position is an island — it
knows only its own token. What's the simplest way to let position `t` *see* the ones before it?

**Idea:** make each position the **average of itself and all earlier positions**. Position 0
sees only itself; position 4 sees the mean of tokens 0–4; etc. It's a crude "context so far"
summary — but crucially it only looks *backward* (a token can't see its own future, or
generation would be cheating).

We'll build this three ways and prove they're equal:
1. explicit loop (slow, obvious),
2. a lower-triangular matrix multiply (the "aha" — a masked weighted average *is* a matmul),
3. softmax over a masked score matrix (which is exactly self-attention with all-equal weights).

That third form is one small step from real attention: instead of *fixed equal* weights,
let the tokens **decide** how much to attend to each other — that's Q/K/V.

_(Next session we scaffold this. Setup above is ready.)_

In [7]:
# A tiny toy tensor to SEE the averaging. Not embeddings yet -- just numbers.
# B,T,C here are stand-ins: 4 sequences, 8 positions, 2 "features" per position.
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
print('x', x.shape)

# GOAL: xbow[b, t] = average of x[b, 0..t]  (the "bag of words so far", backward-only)

# --- Version 1: explicit loop (ground truth) ---
# TODO(human):
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]        # (t+1, C): this position and all before it
        xbow[b, t] = xprev.mean(dim=0)          # mean over the time axis -> (C,)

# --- Version 2: the same thing as ONE matrix multiply ---
# The insight: a backward-only average is (weights) @ x, where `wei` is (T, T),
# row t is uniform over columns 0..t and sums to 1 (zeros for the future).
# TODO(human):
wei = torch.tril(torch.ones(T, T))     # lower-triangular ones: allow past+present, block future
wei = wei / wei.sum(dim=1,keepdim=True)                        # normalize each ROW to sum to 1 (hint: sum over dim=1, keepdim=True)
xbow2 = wei @ x                        # (T,T) @ (B,T,C) -> broadcasts to (B,T,C)

# --- Proof they match ---
print('close?', torch.allclose(xbow, xbow2))


x torch.Size([4, 8, 2])
close? True


In [8]:
# --- Version 3: same average, but via masked scores + softmax ---
# This is the exact machinery real attention uses to hide the future.

tril = torch.tril(torch.ones(T, T))     # 1s on/below diagonal, 0s above (the future)
wei = torch.zeros((T, T))               # start from EQUAL scores (all zero)

# TODO(human): two lines.
# 1) Kill the future: wherever tril == 0, replace wei with a value that softmax
#    will turn into 0 weight. (What input makes exp() -> 0? There's a Tensor method
#    for "set these positions to X where a mask is true".)
# 2) Turn each row of scores into a distribution over the allowed positions.

wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=1)


print(wei)                              # after your two lines: rows = 1/(t+1) over the past, 0 in the future
xbow3 = wei @ x
print('close?', torch.allclose(xbow, xbow3))


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
close? True


## Step 8 — Single-head self-attention (Q / K / V)

Now the kernel stops being a *box* blur. Each token projects its embedding into **three**
vectors:

- **query (q)** — "what am I looking for?"
- **key (k)** — "what do I contain / advertise?"
- **value (v)** — "what do I hand over if you attend to me?"

The affinity between position *i* and position *j* is the dot product `q_i · k_j` — how well
*i*'s query matches *j*'s key. Those dot products *are* the (now content-dependent) kernel
weights. Fill a `(T,T)` matrix with all of them, apply the **same causal mask + softmax** you
just built, then blur the **values**: `out = wei @ v`.

Two "why"s to hold:
- **Why a separate `value`, not just `x`?** What a token contributes when attended to is a
  *learned projection* of itself — q/k decide *how much* to attend, v decides *what flows*.
- **Blur analogy:** the box blur used one fixed kernel; here each position looks at the others
  (`q·k`) and builds *its own* kernel, then blurs `v` with it. A content-aware, causal,
  spatially-varying filter.

Shapes:
```
x            (B, T, C)
q, k, v      (B, T, head_size)     via three Linear(C, head_size)
wei = q@kᵀ   (B, T, T)             every position's affinity with every allowed other
out = wei@v  (B, T, head_size)
```
_(There's a √head_size scaling refinement we'll add later — ignore it for now.)_

In [9]:
# Single-head self-attention. The blur kernel is now COMPUTED from the tokens.
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)          # pretend these are token embeddings

head_size = 16
# The three projections (given). Each maps a C-dim token -> a head_size-dim vector.
key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

# Project x into keys, queries, values (given -- just applying the layers):
k = key(x)      # (B, T, head_size)
q = query(x)    # (B, T, head_size)
v = value(x)    # (B, T, head_size)

# TODO(human):
# 1) AFFINITIES. wei[b,i,j] = dot(q_i, k_j). You have q (B,T,hs) and k (B,T,hs);
#    a matmul q @ k needs k's last two dims swapped to line up -> result (B, T, T).
#    (Method that swaps two axes of a tensor: .transpose(dim0, dim1). Swap the last two.)
# wei = ...
wei = q @ k.transpose(1,2)

# 2) CAUSAL MASK + SOFTMAX. Exactly what you did in Version 3, but now `wei` is computed,
#    not zeros. Mask the future to -inf, then softmax the rows.
#    (Heads-up: wei is 3-D (B,T,T) now, so pick the row-normalizing dim accordingly.)
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, 2)

# 3) GATHER VALUES. Blur v with the kernel -> out (B, T, head_size).
out = wei @ v

print('wei[0]\n', wei[0])     # NOT uniform anymore -- a learned, per-row distribution
print('out', out.shape)       # expect (B, T, head_size) = (4, 8, 16)

wei[0]
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)
out torch.Size([4, 8, 16])


## Step 9 — Wrap it in an `nn.Module` (a reusable `Head`)

The toy cell proved the mechanism. Now make it a **layer** — an object that owns its own
`query`/`key`/`value` weights, moves to the device with `.to(device)`, and can be dropped into
a model and trained. Same four lines of math you just wrote, now living inside `forward`.

Two new PyTorch wrinkles you'll meet here:

- **`register_buffer('tril', ...)`** — `tril` is a tensor the module needs at runtime, but it's
  **not a parameter**: there's nothing to learn, no gradient. A buffer is exactly that — non-learned
  state that still *rides along* with the module (`.to(device)` moves it, `state_dict` saves it).
  If you made it a plain `self.tril = ...`, it would stay on CPU when the model went to MPS and you'd
  get a device-mismatch crash mid-forward.
- **Slicing the mask `self.tril[:T, :T]`** — the buffer is built at max size `(block_size, block_size)`,
  but at **generation time** the actual sequence length `T` can be *smaller* than `block_size`. So you
  crop the mask to the live `T`. This is the same "read `T` from the tensor, never hardcode it" lesson
  that bit you in Part 1's loss reshape — same principle, new place.

And the refinement we deferred:

- **Scale by `1/√head_size`** right after `q @ kᵀ`. Without it, the dot products grow with `head_size`,
  softmax sees large values, saturates toward one-hot, and gradients vanish. The `√head_size` divisor
  keeps the score variance ~1. (Derivation available on request — for now, just apply it.)

Everything except the `forward` math is given below. Your turn is the four-ish lines you already
know, plus those two wrinkles baked in.

In [10]:
# Step 9 — the head as a reusable layer.
n_embd = 32          # embedding dimension (this was "C" in the toy)

class Head(nn.Module):
    """ One head of causal self-attention. """

    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # Non-learned state that must ride the device + persist -> buffer, not parameter.
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.head_size = head_size

    def forward(self, x):
        B, T, C = x.shape        # C == n_embd
        k = self.key(x)          # (B, T, head_size)
        q = self.query(x)        # (B, T, head_size)
        v = self.value(x)        # (B, T, head_size)

        # TODO(human): the same math from the toy cell, now with two wrinkles baked in.
        #   1) wei = q @ kᵀ, then SCALE by self.head_size ** -0.5   (the 1/√head_size divisor)
        #   2) mask the future with self.tril[:T, :T]  (cropped to the live T, not block_size)
        #   3) softmax over the last axis (dim=-1)
        #   4) out = wei @ v      -> (B, T, head_size)
        # out = ...

        wei = q @ k.transpose(1,2) * (self.head_size ** -0.5)
        wei = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf'))
        wei = F.softmax(wei, 2)

        out = wei @ v
        return out


# --- Smoke test: same toy input, but through the module ---
torch.manual_seed(1337)
x = torch.randn(4, block_size, n_embd)   # (B=4, T=block_size=8, n_embd=32)
head = Head(head_size=16)
out = head(x)
print('out', out.shape)                  # expect (4, 8, 16)
print('finite?', torch.isfinite(out).all().item())   # no NaNs from the -inf rows


out torch.Size([4, 8, 16])
finite? True


## Step 10 — Assemble the single-head attention language model

Glue everything together — token identity, position, your `Head`, and a projection back to
characters — then train it and watch it beat the bigram's 2.5 ceiling. First "it's alive" moment.

**Two things that are genuinely new (and easy to get wrong):**

- **Position embedding is not optional.** Attention is order-blind: `q·kᵀ` gives the same
  affinities no matter how the tokens are ordered (the mask hides the *future*, but among visible
  tokens it can't tell *sequence*). A separate position table is the *only* thing that tells the
  model "this token is 3rd, that one is 5th." Forget it and it can never learn word order.

- **Token and position get summed.** Each token's vector = "what I am" + "where I am", both in the
  same `n_embd` space. The position vectors are the same for every sequence in the batch, so the
  add relies on broadcasting.

The `__init__` (the four layers you'll wire), the loss reshape, and `generate` are given. Your
turn is the `forward`. Peek at `generate`'s `idx[:, -block_size:]` crop and make sure you see
*why* it's there — it's the same family as the `self.tril[:T,:T]` slice you already wrote.

In [11]:
# Step 10 — the single-head attention language model.

class AttentionLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_head = Head(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # TODO(human): build the forward pass -> produce `logits` of shape (B, T, vocab_size).
        # Wire the four layers from __init__. Don't forget position.

        embeddings = self.token_embedding_table(idx)
        positions = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = embeddings + positions

        x = self.sa_head(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) in the current context
        for _ in range(max_new_tokens):
            # CROP to the last block_size tokens: the position table only has block_size
            # rows, so a longer context would index past its end. (Sibling of tril[:T,:T].)
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)         # (B, t, vocab)
            logits = logits[:, -1, :]          # focus on the last step -> (B, vocab)
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)   # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)              # grow along time
        return idx


In [12]:
# Instantiate, move to device, train, then sample. (All given -- reuse of Part 1's ritual.)
torch.manual_seed(1337)
model = AttentionLM().to(device)
print('params:', sum(p.numel() for p in model.parameters()))

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(5000):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 500 == 0:
        print(f'step {step:4d} | loss {loss.item():.4f}')

# Sample 300 chars starting from a single 0 (newline) token.
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print('\n--- sample ---')
print(decode(model.generate(context, max_new_tokens=300)[0].tolist()))


params: 7553
step    0 | loss 4.1555
step  500 | loss 3.0707
step 1000 | loss 3.0915
step 1500 | loss 2.4581
step 2000 | loss 2.6892
step 2500 | loss 2.4533
step 3000 | loss 2.7828
step 3500 | loss 2.0234
step 4000 | loss 2.6257
step 4500 | loss 2.6313

--- sample ---

Thol an?

Therat st tey afs omy hi sk ndrod dianfim ypsong me prst mime wist woonid:
W:
K

NUKALHAk-apyord swhut wes by minland inod:
NOu baneinge ors Vaby whive por faktes stes th youdousttisol ne a pr EB me, sd Gm ve?
forand fr y; ro al t it lll, arlm: I m wing ot,
Wore whir cit or ay ain
Torde th
